In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langchain_ollama import ChatOllama

model = ChatOllama(model="qwen3:14b", temperature=0)

In [3]:
from langchain.tools import tool, ToolRuntime

@tool
def read_email() -> str:
    """Read the email from the current state. ALWAYS call this first before responding."""
    # Access email from state via the agent context
    return "EMAIL_PLACEHOLDER"  # Will be overridden by state injection

@tool  
def send_email(body: str, recipient: str = "John") -> str:
    """Send an email response. Requires human approval before execution."""
    return f"✅ Email sent to {recipient}: {body}"

In [4]:
from langgraph.prebuilt import ToolNode
from typing import TypedDict, Annotated
from langgraph.graph.message import add_messages
from langchain.agents import create_agent, AgentState
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import HumanInTheLoopMiddleware

class EmailAgentState(AgentState):
    messages: Annotated[list, add_messages]
    email: str


def create_email_tools(email: str):
    @tool
    def read_email() -> str:
        """Read the email from the current state."""
        return email
    
    @tool  
    def send_email(body: str, recipient: str = "John") -> str:
        """Send an email response. Requires human approval."""
        return f"✅ Email sent to {recipient}: {body}"
    
    return [read_email, send_email]


# Get email from input
email_content = "Hi Seán, I'm going to be late for our meeting tomorrow. Can we reschedule? Best, John."

agent = create_agent(
    model=model,
    tools=create_email_tools(email_content),  # Tools with email injected
    state_schema=EmailAgentState,
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "read_email": False,
                "send_email": True,
            },
            description_prefix="⚠️ Tool execution requires approval",
        ),
    ],
)

In [5]:
from pprint import pprint
from langchain_core.messages import HumanMessage, SystemMessage

config = {"configurable": {"thread_id": "1"}}


input_data = {
    "messages": [
        SystemMessage(content="You are an email assistant. Read the email first, then draft and send a response."),
        HumanMessage(content="Please read my email and send a response.")
    ]
    # Note: email is now in the tools, not state
}

print("🚀 Starting agent execution...\n")

for event in agent.stream(input_data, config=config, stream_mode="updates"):
    pprint(event)
    print("-" * 60)

🚀 Starting agent execution...

{'model': {'messages': [AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen3:14b', 'created_at': '2026-03-14T21:57:15.985124315Z', 'done': True, 'done_reason': 'stop', 'total_duration': 379709797055, 'load_duration': 33957476232, 'prompt_eval_count': 198, 'prompt_eval_duration': 78306418219, 'eval_count': 488, 'eval_duration': 266779153114, 'logprobs': None, 'model_name': 'qwen3:14b', 'model_provider': 'ollama'}, id='lc_run--019cee55-2b5f-73c2-be9f-71b1c8963f1c-0', tool_calls=[{'name': 'read_email', 'args': {}, 'id': '62904eed-301d-4b12-868f-7f00da7e8a53', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 198, 'output_tokens': 488, 'total_tokens': 686})]}}
------------------------------------------------------------
{'HumanInTheLoopMiddleware.after_model': None}
------------------------------------------------------------
{'tools': {'messages': [ToolMessage(content="Hi Seán, I'm going to be late for ou

In [6]:
state = agent.get_state(config)
interrupts = state.metadata.get("interrupts", [])

if interrupts:
    print("\n⚠️  EXECUTION INTERRUPTED!")
    print(f"Interrupt details: {interrupts}")
    
    print("\n📋 Current State:")
    pprint(state.values)
    
    # Resume execution after human approval
    print("\n✅ Resuming execution after approval...\n")
    for event in agent.stream(None, config=config, stream_mode="updates"):
        pprint(event)
        print("-" * 60)
else:
    print("\n✅ Execution completed without interrupts")


✅ Execution completed without interrupts


In [7]:
final_state = agent.get_state(config)
print("\n📬 FINAL STATE:")
pprint(final_state.values)


📬 FINAL STATE:
{'email': "Hi Seán, I'm going to be late for our meeting tomorrow. Can we "
          'reschedule? Best, John.',
 'messages': [SystemMessage(content='You are an email assistant. Read the email first, then draft and send a response.', additional_kwargs={}, response_metadata={}, id='031750db-0bde-48d6-b186-c5e99f3d8a98'),
              HumanMessage(content='Please read my email and send a response.', additional_kwargs={}, response_metadata={}, id='2da255af-981f-4947-88e3-e7c74c99cfa7'),
              AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen3:14b', 'created_at': '2026-03-14T21:38:43.8543731Z', 'done': True, 'done_reason': 'stop', 'total_duration': 126247759353, 'load_duration': 238417262, 'prompt_eval_count': 213, 'prompt_eval_duration': 466561431, 'eval_count': 248, 'eval_duration': 125366417261, 'logprobs': None, 'model_name': 'qwen3:14b', 'model_provider': 'ollama'}, id='lc_run--019cee47-c1a8-7d30-9d6f-12185c52c916-0', tool_calls=[{'

In [ ]:
print(response['__interrupt__'])

In [ ]:
# Access just the 'body' argument from the tool call
print(response['__interrupt__'][0].value['action_requests'][0]['args']['body'])

## Approve

In [ ]:
from langgraph.types import Command

response = agent.invoke(
    Command( 
        resume={"decisions": [{"type": "approve"}]}
    ), 
    config=config # Same thread ID to resume the paused conversation
)

pprint(response)

## Reject

In [ ]:
response = agent.invoke(
    Command(        
        resume={
            "decisions": [
                {
                    "type": "reject",
                    # An explanation of why the request was rejected
                    "message": "No please sign off - Your merciful leader, Seán."
                }
            ]
        }
    ), 
    config=config # Same thread ID to resume the paused conversation
    )   

pprint(response)

In [ ]:
response['messages'][-3].tool_calls[0]['args']['body']

## Edit

In [ ]:
response = agent.invoke(
    Command(        
        resume={
            "decisions": [
                {
                    "type": "edit",
                    # Edited action with tool name and args
                    "edited_action": {
                        # Tool name to call.
                        # Will usually be the same as the original action.
                        "name": "send_email",
                        # Arguments to pass to the tool.
                        "args": {"body": "This is the last straw, you're fired!"},
                    }
                }
            ]
        }
    ), 
    config=config # Same thread ID to resume the paused conversation
    )   

pprint(response)